# SarcasTone - Notebook 00: Setup & Data Verification

**Goal.** Prepare a GPU Colab runtime to reproduce every reported Phase 1/2 number, and
verify the *locked* evaluation contract (same train/val/test for every experiment).

**Locked contract.** MUStARD++ 690 utterances, stratified seed 42 -> train 482 / val 104 / test 104.
Select models on **val**; touch **test once**. Official metric = macro-F1.

Run order: `00_setup` -> `01_text` -> `02_speech`.

In [ ]:
import sys, platform
print('python', sys.version.split()[0], '|', platform.platform())
try:
    import torch
    print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
          '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')
except Exception as e:
    print('torch not installed yet:', e)

## 1. Clone the repo and install dependencies

`PROJECT_ROOT` is discovered from `src/sarcastone/utils.py`, so all paths are CWD-independent.

In [ ]:
import os, sys
REPO_URL = 'https://github.com/PShashankreddy/SarcasTone.git'
PROJECT  = '/content/SarcasTone'
if not os.path.isdir(os.path.join(PROJECT, 'src', 'sarcastone')):
    !rm -rf $PROJECT
    !git clone $REPO_URL $PROJECT
os.chdir(PROJECT)
sys.path.insert(0, os.path.abspath('src'))   # import works even if pip -e is skipped
print('cwd =', os.getcwd())
print('src on path =', os.path.abspath('src'))

In [ ]:
# Colab already ships torch + CUDA. Install the rest, then the package itself.
!pip -q install -r requirements.txt
!pip -q install -e .
print('installed.')

In [ ]:
import sarcastone
from sarcastone.utils import (SPLITS_DIR, REPORTS_DIR, FEATURES_SEQ_DIR,
                              SUMMARY_CSV, CKPT_DIR, SEED)
print('sarcastone  :', sarcastone.__file__)
print('SEED        :', SEED)
print('splits dir  :', SPLITS_DIR)
print('features dir:', FEATURES_SEQ_DIR)
print('reports dir :', REPORTS_DIR)
print('ckpt dir    :', CKPT_DIR)

## 2. Verify the locked splits

The split CSVs are committed to the repo. The **test/val SHAs must never change** -
they are the single source of truth every model is scored against. Record the printed
`sha16` values; if any later run changes them, the result is invalid.

In [ ]:
import hashlib
from pathlib import Path
import pandas as pd

def sha16(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

locked = {}
for s in ('train', 'val', 'test'):
    p  = Path(SPLITS_DIR) / f'{s}.csv'
    df = pd.read_csv(p)
    locked[s] = {'n': len(df), 'sha16': sha16(p)}
    print(f'{s:5s} n={len(df):3d}  sarc={int((df.label==1).sum()):3d}  '
          f'non={int((df.label==0).sum()):3d}  sha16={sha16(p)}')

assert locked['val']['n'] == 104 and locked['test']['n'] == 104, 'locked sizes changed!'
print('\nlocked val/test sizes OK (104 / 104).')

## 3. Pull the Git-LFS model checkpoints (required for 01 and E1/E2)

Trained checkpoints are tracked with Git LFS (`checkpoints/text_*`). A clone without LFS
leaves tiny **pointer** files - loading them raises `SafetensorError: header too large`.
This cell pulls the real weights. Verify the printed sizes are hundreds of MB.

In [ ]:
import shutil, os
LFS_FILE = 'checkpoints/text_roberta_nh/model.safetensors'
def is_pointer(p, threshold=100_000):
    # a Git-LFS pointer is a tiny text file; real weights are hundreds of MB
    return (not os.path.exists(p)) or os.path.getsize(p) < threshold
if is_pointer(LFS_FILE):
    print('pulling LFS checkpoints (first run only)...')
    if shutil.which('git-lfs') is None:
        !apt-get -qq install -y git-lfs > /dev/null 2>&1
    !git lfs install --skip-repo
    !git lfs pull
else:
    print('LFS checkpoints already present.')
!ls -lh checkpoints/text_roberta_nh/model.safetensors checkpoints/text_roberta_boosted/model.safetensors

## 4. Data provenance

| Asset | Where | Committed? | Needed for |
|---|---|---|---|
| `data/processed/splits/*.csv` | repo | yes | every experiment (locked contract) |
| `data/processed/dataset.csv` | repo | yes | text rows |
| `data/processed/features_seq/*.npy` | repo | yes | Phase 2 (acoustic sequences) |
| `data/processed/acoustic_summary.csv` | repo | yes | Phase 2 LR baseline |
| `data/raw/sarcasm_data.json` | public GitHub | no | re-deriving splits |
| `data/raw/audio/*.wav` | collected | no | only to **rebuild** features |

Because the feature tensors and summary are committed, **all reported Phase-2 numbers
are reproducible without the raw audio**. Audio is only needed to regenerate features.

### Caveats (kept explicit, not hidden)
- MUStARD++ *full* (1202) `context` is rebuilt by joining CSV `SCENE_c_NN` rows; it matches the
  original `sarcasm_data.json` context ~95.5% on overlapping scenes. Locked 690 are never altered.
- The 514 expansion clips were mirrored from a public HF upload and verified by filename/count
  against the official Google Drive listing. Cite the official source in any write-up.

In [ ]:
# Annotation file for the LOCKED 690 (public, MIT). Re-derives the same splits from seed 42.
from sarcastone.data.download_mustard import download_mustard
download_mustard()

## 5. Optional: raw audio (only to rebuild acoustic features)

Skip this unless you intend to recompute `features_seq`. Keep `FETCH_AUDIO=False` to just
reproduce the committed results.

In [ ]:
FETCH_AUDIO = False   # set True to download the public MUStARD++ mirror

if FETCH_AUDIO:
    !pip -q install huggingface_hub
    from huggingface_hub import snapshot_download
    snapshot_download('sifan077/MUStARD_plus_plus', repo_type='dataset',
                      local_dir='data/raw/mustard_pp/_zips')
    print('\nNext: extract the *_u.mp4 clips into data/raw/audio_raw/, then run')
    print('  python -m sarcastone.features.convert_audio      # -> 16k mono WAV')
    print('  python -m sarcastone.features.build_features     # -> features_seq + summary')
else:
    print('skipped - committed feature tensors are sufficient for 02_speech.')

## 6. Optional: News-Headlines booster corpus (Kaggle token required)

Only needed to retrain the booster from scratch (`configs/boost_headlines.yaml`). The
already-boosted checkpoint `checkpoints/text_roberta_nh` ships via LFS.

In [ ]:
FETCH_HEADLINES = False

if FETCH_HEADLINES:
    from google.colab import files
    files.upload()  # choose kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    # place Sarcasm_Headlines_Dataset.json in data/raw/ first, then:
    !python -m sarcastone.data.news_headlines
else:
    print('skipped.')

## Setup complete

Next: open **`01_text.ipynb`** (Phase 1 text ladder + champion), then **`02_speech.ipynb`**.